In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`ET` data is not connected (see [](general:ic)). No joining process is necessary, as the data is in the longitudinal format.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et"]
assert len(split_data(data, idcols)) == 2, "Not 2 different row types present!?"

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). We kept all rows.

In [ ]:
data["Institute with a start date"] = (
    (~data["start_date_dso"].isna()) + (~data["start_date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
data["start_date"] = collapse_col(
    data.loc[:, ["start_date_dso", "start_date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "start_date",
    "Institute with a start date",
)
data.drop(
    columns=["start_date", "donor", "Institute with a start date"],
    inplace=True,
)

### Unit Conversions

We applied the common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

We consolidated columns that appear for {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `start_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["start_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
class DonorPostmortemMedication(SpenderID):
    antibiotics_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Reason for antibiotics",
        description="Why were antibiotics prescribed?",
        isin=[
            "prophylaktisch",
            "kalkuliert (klinische Symptome)",
            "gezielt (Keimnachweis)",
        ],
    )
    antibiotics_text: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Antiobiotics",
        description="Which antibiotic was prescribed?",
    )
    backtable: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Backtable Flush",
        description="Was a backtable flush performed?",
        isin=["no", "in situ", "ex situ backtabel"],
    )
    bloodgroup: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bloodgroup",
        description="The patients bloodgroup",
        isin=["A", "0", "B", "AB"],
    )
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    dosage: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dosage",
        description="Dosage of the medication",
    )
    dosage_unit: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dosag Unit",
        description="Unit of the dosage column",
    )
    end_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="End Date",
        description="Date when the medication was no longer provided",
    )
    medication_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Medication Type",
        description="How was the medication provided?",
        isin=[
            "intravenös",
            "Perfusor",
            "subkutan",
            "oral",
            "Pulmonalarterie",
            "Magensonde",
            "intramuskulär",
            "rektal",
            "rechtes Atrium",
        ],
    )
    name: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Medication Name",
        description="Which medication was provided?",
    )
    other_bloodproduct: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Blood product Name",
        description="Which blood product was provided?",
    )
    perfusion_location: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Perfusion location",
        description="Were was the perfusion?",
        isin=["aortal", "Herz", "Lunge", "portal"],
    )
    plasma_note: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Plasma Product provided",
        description="Which plasma product was provided?",
    )
    start_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Start Date",
        description="Date when the medication was provided",
    )
    transfusion: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Transfusion provided",
        description="Which transfusion product was provided?",
    )

    class Config:
        title = "Donor Postmortem Medication Dataset"
        description = "Each row represents a medication provided to a deceased donor. The data is based on the 'element_spender_postmortem_medikation.csv' file. It contains data from the DSO and ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemMedication, data)

In [ ]:
DonorPostmortemMedication.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemMedication.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)